# Solutions — Event handling

Only look here after you've actually tried the exercises in `events.ipynb`.

### LESSON 22 — Exercise

In [ ]:
function l22describe(value) {
  return typeof value === "function" ? "a handler" : "nothing to call";
}

function l22named() {
  return "done";
}

console.log("named function:", l22describe(l22named));
console.log("arrow:         ", l22describe(() => l22named()));
console.log("called result: ", l22describe(l22named()));
console.log("a string:      ", l22describe("handleClick"));

`l22named()` is the one to look at twice: it is a perfectly good function, and passing
`l22named()` still leaves React with `"done"` — a string. The mistake is never the function,
it is the two characters after it.

**Part 2 — a function that makes a function.**

In [ ]:
function l22makeGreeter(name) {
  return function () {
    console.log("hello,", name);
  };
}

console.log("creating two greeters (nothing should be logged by this line)");
const l22greetAda = l22makeGreeter("Ada");
const l22greetLinus = l22makeGreeter("Linus");

console.log("now calling them:");
l22greetAda();
l22greetLinus();

// Part 3 — `onClick={handleGreet("Ada")}` CALLS handleGreet during render. React is handed
// its return value, which for a logging function is undefined. There is no function left to
// call on click, so the button does nothing. `onClick={() => handleGreet("Ada")}` passes an
// arrow instead; the arrow is the function React keeps, and it calls handleGreet later.

**Common mistakes.**

- Writing `l22describe(value) { return value === "function" }` — comparing the value to a
  string instead of using `typeof`.
- Returning the greeting instead of logging it in Part 2, then wondering why nothing appears.
  A click handler's return value is discarded; if you want to see something, log it.
- Building the greeter with `l22makeGreeter("Ada")()` — calling it immediately, which is the
  very mistake the exercise is about.

### LESSON 22 — Mini challenge

**1. What was already logged.** `handleClick ran`, **twice**, before any click.

The `onClick={handleClick()}` on the third button runs during render. It appears twice
because `<StrictMode>` deliberately renders components an extra time in development
(LESSON 3) — so a handler that is wrongly called shows up as a *pair* of log lines, which is
a useful tell.

**2. What each button logs when clicked.**

```text
#ok     -> handleClick ran
#arrow  -> hello, Ada
#broken -> (nothing)
```

**3. Why the broken one is silent.** `handleClick()` returned `undefined`, so
`onClick={undefined}` is what React received. React attaches no listener at all, and a
button with no listener is just a button. There is nothing to throw, because nothing is ever
called.

Compare this with LESSON 16, where the same mistake *did* throw
`TypeError: onSelect is not a function`. The difference is who does the calling: there, your
own component called the prop, so `undefined()` blew up. Here React does the calling, and
React checks first. Same mistake, two very different symptoms — which is exactly why it is
worth recognising the cause rather than the error message.

**4. The fix.**

```jsx
<button onClick={handleClick}>Passed correctly</button>
<button onClick={() => handleGreet("Ada")}>Arrow passes an argument</button>
<button onClick={handleClick}>Fixed</button>
```

The middle button was never broken: an arrow is a value, so writing it does not run it. That
is precisely why the arrow is the tool for passing arguments — it delays the call until the
click.

### LESSON 23 — Exercise

All of this happens in `playground/src/experiments/07-event-object.jsx`, so the answers are
written out.

**1. Removing `preventDefault()`.** The console still logs `link clicked, navigation
prevented` — your handler runs first — and then the browser leaves the page for
`example.com`. The order is the point: `preventDefault()` does not stop *your* handler, it
cancels what the **browser** was going to do afterwards.

**2. Who owns the text.** The browser does. `<input onChange={handleTyping} />` has no
`value` prop, so React never tells it what to display; the DOM keeps the text and React only
listens. That is an *uncontrolled* input. Handing React the value is topic 10.

**3. The order, and why.**

```text
inner button handler
  outer div handler
```

Innermost first, then outwards. The event starts at the element you actually clicked and
travels up through its ancestors, running each handler it meets. The button's handler is
therefore always first.

**4. Removing `stopPropagation()`.** The outer div's line comes back, because the event is
no longer being halted at the button.

**5. `target` versus `currentTarget`** inside the outer div's handler, having clicked the
inner button:

```text
inner   outer
```

`event.target` is `inner` — the button, where the click actually happened.
`event.currentTarget` is `outer` — the element whose handler is running. Same event, two
different questions: *where did this start* and *who is handling it now*.

**Common mistakes.**

- Expecting `preventDefault()` to stop the outer handler. It does not touch propagation at
  all — different job, different method.
- Reading `event.target.value` in the **outer** handler after clicking a button. `target` is
  the button, and a button has no meaningful `value` — this is exactly why LESSON 24 uses
  `currentTarget` and data attributes instead.
- Commenting out `preventDefault()` and concluding the handler never ran because the page
  navigated away. It ran; the log is simply lost with the page. Tick "Preserve log" in the
  console if you want to see it.

### LESSON 23 — Mini challenge

**1. The clickable row with a Delete button.** Call `event.stopPropagation()` **inside the
Delete button's own handler**, as the first thing it does. The event then never reaches the
row's handler, so the record does not open.

Putting it in the row's handler would be too late — by the time the row runs, the button has
already had its turn, and stopping propagation there would only affect anything *above* the
row.

**2. One and not the other.**

- **`preventDefault()` without `stopPropagation()`:** a link inside a card, where you want to
  handle the navigation yourself but the card still needs to know it was clicked. Cancel the
  browser's navigation; let the event carry on upwards.
- **`stopPropagation()` without `preventDefault()`:** a checkbox inside a clickable row. The
  checkbox *should* still tick — that is the browser doing exactly the right thing — but the
  row must not open. Keep the default, stop the bubbling.

They are unrelated: one is about the browser's built-in behaviour, the other about which
handlers see the event.

**3. Why `<input value={event.target.value} />` cannot work.** There is no `event` at that
point. `event` exists only *inside* a handler, while it is running; in the JSX there is
nothing of that name, so the line is a `ReferenceError` waiting to happen.

The deeper problem is that the value has nowhere to live. To display what was typed, the
component must remember it between renders — and remembering is exactly what **state**
(topic 9) is for. Controlled inputs, which is what they are reaching for, are topic 10 and
are built on it.

### LESSON 24 — Exercise

In [ ]:
const l24filters = {
  all: "showing everything",
  unpaid: "showing unpaid invoices",
  overdue: "showing overdue invoices",
};

function l24describeFilter(name) {
  return l24filters[name] ?? `unknown filter: ${name}`;
}

for (const name of ["all", "unpaid", "overdue", "archived"]) {
  console.log(name.padEnd(9), "->", l24describeFilter(name));
}

`??` rather than `||` for the same reason as LESSON 13: `||` would also replace a legitimate
empty string, and "the lookup found nothing" is exactly what `??` tests for.

**Part 2 — reading from the right place.**

In [ ]:
function l24readAction(eventLike) {
  return eventLike.currentTarget.dataset.action;
}

const l24clickOnSpanInsideButton = {
  currentTarget: { dataset: { action: "export" } }, // the button we wired up
  target: { dataset: {} },                          // the span that was actually clicked
};

console.log("from currentTarget:", l24readAction(l24clickOnSpanInsideButton));
console.log("from target:       ", l24clickOnSpanInsideButton.target.dataset.action);

// Part 3 — currentTarget keeps working, because it is always the element the handler is
// attached to. target breaks: it becomes the icon span, which has no data-action, so the
// value is undefined and the dispatch silently does nothing.

**Common mistakes.**

- Writing the lookup as `if (name === "all") ... else if ...`. It works and it grows badly;
  the exercise asks for the lookup because that is the shape that stays short.
- Forgetting the unknown case, so `l24describeFilter("archived")` returns `undefined` and the
  UI renders the word "undefined".
- Using `l24filters[name] || ...` — fine here, wrong the moment a filter's description is
  legitimately an empty string.

### LESSON 24 — Mini challenge

**Part 1 — what the playground shows.**

1. Each button reports its own action:

```text
action: refresh | target was: BUTTON
action: export  | target was: BUTTON
action: archive | target was: BUTTON
```

2. Clicking the text inside a button:

```text
action: refresh | target was: SPAN
```

The action is **still correct**. `target` changed to the span, because that is where the
click landed, but `currentTarget` is still the button — the element whose handler is running.
The button is where `data-action` lives, so the lookup is unaffected.

3. **Switching to `event.target`** and clicking the span gives:

```text
action: undefined
```

The span has no `data-action` attribute, so `dataset.action` is `undefined` and the dispatch
finds nothing. This is the bug that appears the day a designer adds an icon inside a button
that worked perfectly for months — and it is the whole reason the lesson prefers
`currentTarget`.

**Part 2 — judgement.**

1. **Five formatting buttons — one handler.** They are five variations of a single job
   ("apply a formatting command"), differing only by name. A sixth is a data entry, not a
   code change.

2. **Save draft / Open settings / Log out — three handlers.** These share nothing but a
   container. One handler would have to know about persistence, dialogs and authentication
   at once, and reading it would tell you less than reading three small functions. Merging
   them creates a switchboard, not a reuse.

3. **What one application-wide handler would cost.** Every action becomes reachable from one
   function, so that function grows without limit and eventually depends on half the app. The
   connection between a button and its behaviour stops being a reference you can follow —
   your editor's "go to definition" lands on the switchboard, not on the code that runs. And
   a typo in a `data-action` string fails silently, because nothing checks that the name
   exists.

   The pattern earns its keep across a handful of related controls. It stops earning it the
   moment "related" becomes "in the same application".